# Automated Code Review with LangGraph
### Notebook 2 of 3 — Demo & Patterns

Run all four execution paths and explore key observability patterns.

| Notebook | Focus |
|----------|-------|
| 01 — Foundations | State, nodes, graph build |
| **02 — Demo & Patterns** (this one) | All 4 paths, streaming, persistence, token cost |
| 03 — Advanced Patterns | Unit testing, LLM-as-Judge, extensions |

**Prerequisites:** Run the setup cell below, then proceed step by step.

In [ ]:
# ── Setup: import all nodes and rebuild the graph ───────────────
import os, json
from pathlib import Path
from typing import TypedDict, Annotated, Optional, List
import operator

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END

load_dotenv(Path.cwd() / ".env")
NOTEBOOK_DIR = Path.cwd()

# Production modules — all nodes and state defined in these files
from state import (PRReviewState, SimpleReviewOutput, SummarizeFindingsOutput,
                   FixItem, QualityJudgeOutput, merge_usage)
from nodes import (read_pr_file, route_review_type, routing_edge,
                   simple_review_node, quality_analysis_node, security_review_node,
                   aggregate_node, output_guardrail, blocked_review_node,
                   summarize_findings_node, final_decision_node, return_final_answer,
                   full_review_start)

# ── Rebuild graph ────────────────────────────────────────────
builder = StateGraph(PRReviewState)
builder.add_node("read_pr",             read_pr_file)
builder.add_node("route_review",        route_review_type)
builder.add_node("simple_review",       simple_review_node)
builder.add_node("full_review_start",   full_review_start)
builder.add_node("security_review",     security_review_node)
builder.add_node("quality_analysis",    quality_analysis_node)
builder.add_node("aggregate",           aggregate_node)
builder.add_node("blocked_review",      blocked_review_node)
builder.add_node("summarize_findings",  summarize_findings_node)
builder.add_node("final_decision",      final_decision_node)
builder.add_node("return_final_answer", return_final_answer)

builder.add_edge(START, "read_pr")
builder.add_edge("read_pr", "route_review")
builder.add_conditional_edges("route_review", routing_edge,
    {"simple": "simple_review", "full": "full_review_start", "error": "return_final_answer"})
builder.add_edge("full_review_start", "security_review")
builder.add_edge("full_review_start", "quality_analysis")
builder.add_edge("security_review",  "aggregate")
builder.add_edge("quality_analysis", "aggregate")
builder.add_edge("aggregate", "summarize_findings")
builder.add_conditional_edges("summarize_findings", output_guardrail,
    {"proceed": "final_decision", "block": "blocked_review"})
builder.add_edge("blocked_review",  "final_decision")
builder.add_edge("simple_review",   "final_decision")
builder.add_edge("final_decision",      "return_final_answer")
builder.add_edge("return_final_answer", END)

code_review_app = builder.compile()
print("✅ code_review_app ready")
try:
    code_review_app.get_graph().print_ascii()
except Exception:
    pass

## Step 12 — Visualise the Graphs

LangGraph can render both subgraphs and the full graph as PNG using Mermaid. This is invaluable for debugging graph structure before running it.

Two render methods are available:

| Method | Output | When to use |
|--------|--------|-------------|
| `get_graph().print_ascii()` | Terminal ASCII art | Quick structure check without dependencies |
| `get_graph().draw_mermaid_png()` | Inline PNG image | Documentation, sharing, visual verification |

Pass `xray=True` to `get_graph()` to expand inlined subgraphs: `code_review_app.get_graph(xray=True).draw_mermaid_png()`

In [ ]:
try:
    from IPython.display import Image, display
    
    print("=== Parallel Code Review Tool (subgraph) ===")
    display(Image(parallel_code_review.get_graph().draw_mermaid_png()))
    
    print("\n=== Outer Router Graph (full) ===")
    display(Image(code_review_app.get_graph(xray=True).draw_mermaid_png()))
    
except Exception as e:
    print(f"Graph visualisation requires graphviz/mermaid: {e}")
    print("Run: pip install pygraphviz   OR   pip install mermaid-py")

## Step 13 — Run: Full Review Path (`code_changes.txt`)

### What this test file contains

The diff was crafted to trigger every guardrail in the system:

```python
# Vulnerability 1: SQL Injection
query = f"SELECT * FROM users WHERE username = '{username}'"  # unsanitised input

# Vulnerability 2: Plaintext password comparison
if user['password'] == password:  # no hashing

# Vulnerability 3: PII logging
print(f"User {username} logged in")  # username in logs
```

### Expected execution path

```
read_pr → route_review [full] → full_review_start
        → security_review + quality_analysis (parallel)
        → aggregate → output_guardrail [BLOCKING found]
        → blocked_review → final_decision [REJECT]
        → return_final_answer
```

`summarize_findings` is **bypassed** — the output guardrail short-circuits to `blocked_review` when critical security issues are present.

### State persistence

After `invoke()` completes, state is saved to `flow_state.json`:

```python
state_to_save = {k: v for k, v in result_full.items() if k != "messages"}
with open("flow_state.json", "w") as f:
    json.dump(state_to_save, f, indent=2, default=str)
```

We exclude `messages` to keep the JSON readable. In production, store state in a database (LangGraph supports `SqliteSaver` and `PostgresSaver` checkpointers) to enable **resumable flows** and **audit history**.

In [ ]:
print("🚀 Running Code Review Tool on: code_changes.txt")
print("Expected route: full → Parallel Code Review Tool\n")
print("=" * 60)

initial_state: PRReviewState = {
    "pr_file_path":     "files/code_changes.txt",
    "pr_content":       "",
    "review_type":      "",
    "quality_findings": None,
    "security_findings": None,
    "summary":          None,
    "simple_review":    None,
    "final_decision":   None,
    "errors":           [],
    "tokens_used":      {},
    "messages":         []
}

result_full = code_review_app.invoke(initial_state)

# Save full state (excluding verbose messages list) to flow_state.json
state_to_save = {k: v for k, v in result_full.items() if k != "messages"}
with open("flow_state.json", "w") as f:
    json.dump(state_to_save, f, indent=2, default=str)
print("\n💾 State saved to flow_state.json")

print("\n" + "=" * 60)
print("TOKEN USAGE:")
print("=" * 60)
for k, v in result_full.get("tokens_used", {}).items():
    print(f"  {k}: {v}")

## Step 14 — Run: Simple Review Path (`code_changes_simple.txt`)

### What this test file contains

```diff
- import numpy as np
+ import numpy as numpy
```

A single import alias rename — no logic change, no security surface.

### Expected execution path

```
read_pr → route_review [simple] → simple_review → final_decision [APPROVED] → return_final_answer
```

The parallel agents (`security_review`, `quality_analysis`, `aggregate`, `summarize_findings`) are **completely bypassed**. This is the efficiency gain of the router pattern — trivial changes don't pay the cost of a full parallel review.

### Token cost comparison

After running both paths, compare `tokens_used`:

| Path | Input tokens | Output tokens | Total | Relative cost |
|------|-------------|---------------|-------|---------------|
| Full review (blocked) | ~1,978 | ~1,289 | ~3,267 | 4× |
| Simple review | ~689 | ~152 | ~841 | 1× |

The simple path uses roughly **4× fewer tokens** — a significant cost saving at scale when the majority of PRs are trivial changes.

In [ ]:
print("🚀 Running Code Review Tool on: code_changes_simple.txt")
print("Expected route: simple → Simple Review (LLM only)\n")
print("=" * 60)

initial_state_simple: PRReviewState = {
    "pr_file_path":     "files/code_changes_simple.txt",
    "pr_content":       "",
    "review_type":      "",
    "quality_findings": None,
    "security_findings": None,
    "summary":          None,
    "simple_review":    None,
    "final_decision":   None,
    "errors":           [],
    "tokens_used":      {},
    "messages":         []
}

result_simple = code_review_app.invoke(initial_state_simple)

# Save state to flow_state_simple.json
state_to_save = {k: v for k, v in result_simple.items() if k != "messages"}
with open("flow_state_simple.json", "w") as f:
    json.dump(state_to_save, f, indent=2, default=str)
print("\n💾 State saved to flow_state_simple.json")

print("\n" + "=" * 60)
print("TOKEN USAGE:")
print("=" * 60)
for k, v in result_simple.get("tokens_used", {}).items():
    print(f"  {k}: {v}")

## Step 15 — Inspect the Message Log (Audit Trail Pattern)

### Why `messages` is an append-only reducer

Every node appends a brief status string to `state["messages"]`:

```python
return {
    ...,
    "messages": [f"[node_name] brief status"]  # list with one item
}
```

Because `messages` uses `Annotated[list, operator.add]`, each node's single-item list is **concatenated** onto the growing log:

```
[read_pr]          Loaded: code_changes.txt (1140 chars)
[router]           Review type: full
[quality_analysis] complete (3089 chars)
[security_review]  complete (2515 chars)
[blocked]          Review halted by output guardrail — REJECT
[final_decision]   REJECT
```

### What the log reveals

- **Execution order**: `quality_analysis` and `security_review` both appear before `aggregate` — confirming they ran concurrently (LangGraph emits them in completion order, which may vary between runs).
- **Which path ran**: the presence/absence of `[blocked]` vs `[tech_lead_summary]` shows which branch was taken.
- **Node outputs**: character counts and confidence scores provide a quick quality sanity check.

### Extending for production

In production, replace plain strings with structured log entries:

```python
"messages": [{
    "node":      "security_review",
    "status":    "complete",
    "timestamp": datetime.utcnow().isoformat(),
    "chars":     len(content)
}]
```

These can be forwarded to an observability system (Datadog, CloudWatch) for monitoring, alerting, and SLA tracking.

In [ ]:
print("=" * 60)
print("MESSAGE LOG — Full Review Run")
print("=" * 60)
for i, msg in enumerate(result_full["messages"], 1):
    # Print just the first 120 chars of each message to keep it readable
    preview = msg[:120].replace("\n", " ")
    print(f"{i:2}. {preview}..." if len(msg) > 120 else f"{i:2}. {preview}")

print("\n" + "=" * 60)
print("MESSAGE LOG — Simple Review Run")
print("=" * 60)
for i, msg in enumerate(result_simple["messages"], 1):
    preview = msg[:120].replace("\n", " ")
    print(f"{i:2}. {preview}..." if len(msg) > 120 else f"{i:2}. {preview}")

## Step 16 — Streaming: Real-Time Execution Visibility

### Why streaming matters

`graph.invoke()` blocks until the entire graph finishes. `graph.stream()` yields **incremental updates** as each node completes — useful for:

- Showing a live progress indicator to the user during a long review
- Observing parallel execution order in real time (which agent finishes first?)
- Building a real-time UI (stream update events over a WebSocket)
- Debugging: see exactly which node produced which state update

### Reading stream events

```python
for event in code_review_app.stream(state, stream_mode="updates"):
    for node_name, node_output in event.items():
        # node_output = the dict the node returned
        # or None for pass-through nodes (full_review_start, aggregate, return_final_answer)
        if node_output:
            updated_keys = [k for k, v in node_output.items()
                            if v and k not in ("messages", "tokens_used")]
```

`stream_mode="updates"` yields `{node_name: output_dict}` for each node as it completes.

### Pass-through nodes emit `None`

Nodes that return `{}` — like `full_review_start`, `aggregate`, and `return_final_answer` — emit `None` as their output in the stream. Always guard with `if node_output:` before iterating its items.

### Streaming shows parallelism in action

In the output above, `security_review` and `quality_analysis` stream events appear in succession — they ran concurrently but their events arrive as each finishes. The order can differ between runs, which is normal and expected for parallel branches.

In [ ]:
print("🔴 LIVE STREAM — Full Review (code_changes.txt)")
print("=" * 60)

stream_state: PRReviewState = {
    "pr_file_path":     "files/code_changes.txt",
    "pr_content":       "",
    "review_type":      "",
    "quality_findings": None,
    "security_findings": None,
    "summary":          None,
    "simple_review":    None,
    "final_decision":   None,
    "errors":           [],
    "tokens_used":      {},
    "messages":         []
}

for event in code_review_app.stream(stream_state, stream_mode="updates"):
    for node_name, node_output in event.items():
        print(f"\n📍 Node: [{node_name}]")
        if node_output:  # pass-through nodes emit None
            updated_keys = [k for k, v in node_output.items()
                            if v and k not in ("messages", "tokens_used")]
            print(f"   Updated: {updated_keys}")
            if node_output.get("tokens_used"):
                t = node_output["tokens_used"]
                print(f"   Tokens:  in={t.get('input_tokens',0)} out={t.get('output_tokens',0)}")

## Step 17 — Demo: Error Path (File Not Found)

### What this demonstrates

When `read_pr_file` cannot find the file, it stores the error in `state["errors"]` instead of raising an exception. The router detects the populated list and short-circuits **directly to `return_final_answer`** -- no LLM calls, no wasted tokens.

```
read_pr [FileNotFoundError stored in state["errors"]]
    → route_review: state.get("errors") is truthy → return "error"
    → return_final_answer: prints errors, exits cleanly
```

This is the **fourth execution path** wired in Step 11 (`"error": "return_final_answer"`) but not yet run.

### Why this matters for production

In a real CI/CD pipeline, PRs can fail to fetch for many reasons (permissions, branch deleted, API rate limit). Storing errors in state rather than crashing means:
- The graph always reaches a terminal node cleanly
- Error details are captured in `state["errors"]` for logging
- Token costs are zero for unresolvable inputs

In [ ]:
print("🚀 Running Code Review Tool on: nonexistent_file.txt")
print("Expected route: read_pr [ERROR] → route_review [error] → return_final_answer")
print("Expected tokens: 0 (no LLM calls)\n")
print("=" * 60)

error_state: PRReviewState = {
    "pr_file_path":     "files/nonexistent_file.txt",
    "pr_content":       "",
    "review_type":      "",
    "quality_findings": None,
    "security_findings": None,
    "summary":          None,
    "simple_review":    None,
    "final_decision":   None,
    "errors":           [],
    "tokens_used":      {},
    "messages":         []
}

result_error = code_review_app.invoke(error_state)

print("\n" + "=" * 60)
print("ERRORS IN STATE:")
for err in result_error.get("errors", []):
    print(f"  • {err}")

print("\nMESSAGE LOG:")
for msg in result_error.get("messages", []):
    print(f"  {msg}")

print(f"\nTOKEN USAGE (should be empty -- no LLM calls made):")
print(f"  {result_error.get('tokens_used', {})}")

## Step 18 — Demo: Non-Blocking Full Review (`code_changes_needswork.txt`)

### What this demonstrates

Steps 13 and 14 showed the BLOCKING path and the simple path. This cell exercises the **third execution path** -- full review that passes the security guardrail and reaches `summarize_findings`.

`code_changes_needswork.txt` contains a new `analytics/report.py` module with code quality issues but no security vulnerabilities:

| Issue | Type | Severity |
|-------|------|----------|
| `range(len(lst))` in loop | Inefficient -- prefer direct iteration or list comprehension | MINOR |
| Manual `total = total + n` | Use `sum(numbers)` instead | MINOR |
| Verbose two-pass dict merge | Use `{**base, **overrides}` or `base \| overrides` (3.9+) | MINOR |
| Redundant `_is_built` flag | `len(self.rows) > 0` is sufficient | MINOR |

### Expected path

```
read_pr → route_review [full] → full_review_start
        → security_review + quality_analysis (parallel)
        → aggregate → output_guardrail [NON-BLOCKING]
        → summarize_findings [report_guardrail] → final_decision [REQUEST CHANGES]
        → return_final_answer
```

`summarize_findings` now runs -- this is the path that was wired in Step 11 but unreachable with the deliberately broken auth diff.

In [ ]:
print("🚀 Running Code Review Tool on: code_changes_needswork.txt")
print("Expected route: full → NON-BLOCKING → summarize_findings → final_decision\n")
print("=" * 60)

needswork_state: PRReviewState = {
    "pr_file_path":     "files/code_changes_needswork.txt",
    "pr_content":       "",
    "review_type":      "",
    "quality_findings": None,
    "security_findings": None,
    "summary":          None,
    "simple_review":    None,
    "final_decision":   None,
    "errors":           [],
    "tokens_used":      {},
    "messages":         []
}

result_needswork = code_review_app.invoke(needswork_state)

state_to_save = {k: v for k, v in result_needswork.items() if k != "messages"}
with open("flow_state_needswork.json", "w") as f:
    json.dump(state_to_save, f, indent=2, default=str)
print("\n💾 State saved to flow_state_needswork.json")

# Detect which branch was taken
summary_str = result_needswork.get("summary", "") or ""
was_blocked = summary_str.startswith("REJECT")

if was_blocked:
    # output_guardrail returned "block" — summary is plain-text, not JSON
    print("\n⚠️  output_guardrail BLOCKED this review (security findings triggered BLOCKING)")
    print("   Path taken: full → blocked_review → final_decision")
    print(f"\n🚫 Summary preview:\n{summary_str[:300]}")
elif summary_str:
    # summarize_findings ran — summary is a JSON string
    try:
        summary = json.loads(summary_str)
        print(f"\n📊 Tech Lead confidence:  {summary.get('confidence')}/100")
        print(f"📝 Findings:  {summary.get('findings', '')[:250]}")
        fix_items = summary.get("fix", [])
        print(f"\n🔧 Fix items required ({len(fix_items)}):")
        for i, item in enumerate(fix_items, 1):
            print(f"  {i}. {str(item)[:160]}")
    except json.JSONDecodeError as e:
        print(f"\n⚠️  Could not parse summary JSON: {e}")
        print(f"Raw summary: {summary_str[:300]}")

print("\n" + "=" * 60)
print("TOKEN USAGE:")
for k, v in result_needswork.get("tokens_used", {}).items():
    print(f"  {k}: {v}")

## Step 19 — Token Cost Comparison Across All Paths

Now that all four execution paths have been demonstrated, we can compare their token usage directly. This makes the efficiency argument for the router pattern concrete.

| Path | LLM nodes called | Expected cost |
|------|-----------------|---------------|
| Error | 0 | Free |
| Simple | router + simple_review + final | Low |
| Full / BLOCKING | router + 2x parallel + final | Medium (bypasses tech_lead) |
| Full / NON-BLOCKING | router + 2x parallel + tech_lead + final | Highest |

The router's job is to ensure most PRs take the cheapest path that is still safe.

In [ ]:
print("=" * 70)
print(f"{'PATH':<38} {'INPUT':>7} {'OUTPUT':>7} {'TOTAL':>7}  NODES HIT")
print("=" * 70)

paths = [
    ("Error (file not found)",        result_error,      "none -- short-circuits before LLM"),
    ("Simple (import alias rename)",  result_simple,     "router + simple_review + final"),
    ("Full / BLOCKING (SQL injection)", result_full,     "router + 2x parallel + final"),
    ("Full / NON-BLOCKING (quality)", result_needswork,  "router + 2x parallel + tech_lead + final"),
]

for label, result, nodes in paths:
    t   = result.get("tokens_used", {})
    inp = t.get("input_tokens",  0)
    out = t.get("output_tokens", 0)
    tot = t.get("total_tokens",  0)
    print(f"{label:<38} {inp:>7,} {out:>7,} {tot:>7,}  {nodes}")

print("=" * 70)
print(f"\nKey insight: the error path costs 0 tokens.")
print(f"Simple path is ~{result_needswork['tokens_used'].get('total_tokens',1) // max(result_simple['tokens_used'].get('total_tokens',1),1)}x cheaper than full non-blocking review.")

## Step 20 — State Persistence with SqliteSaver (LangGraph Checkpointer)

### What checkpointers do

LangGraph's built-in checkpointers automatically save the full state snapshot after **every node**. This enables:

- **Resumability** -- if a node fails mid-graph, restart from the last good checkpoint instead of from scratch
- **Multi-turn conversations** -- pause a review, get human feedback, then resume with updated state
- **Audit history** -- query all past runs, replay any run, inspect state at any step

### How it works

```python
with SqliteSaver.from_conn_string("checkpoints.db") as memory:
    app = graph_builder.compile(checkpointer=memory)       # attach checkpointer at compile time
    config = {"configurable": {"thread_id": "run-001"}}    # thread_id groups snapshots for one run
    result = app.invoke(state, config)
    # Every node's output is now saved to checkpoints.db automatically
```

The `thread_id` is the key that groups all checkpoints for a single review run. Pass the same `thread_id` to `get_state_history()` to replay or inspect any previous run.

### Checkpoint lifecycle

```
START → read_pr [snapshot 1] → route_review [snapshot 2] → simple_review [snapshot 3] → ...
```

Each snapshot stores the full state at that point -- you can restore to any of them.

In [ ]:
try:
    from langgraph.checkpoint.sqlite import SqliteSaver

    with SqliteSaver.from_conn_string("checkpoints.db") as memory:
        # Recompile the graph with the checkpointer attached
        app_with_memory = outer_builder.compile(checkpointer=memory)

        config = {"configurable": {"thread_id": "pr-review-simple-001"}}

        checkpoint_state: PRReviewState = {
            "pr_file_path":     "files/code_changes_simple.txt",
            "pr_content":       "",
            "review_type":      "",
            "quality_findings": None,
            "security_findings": None,
            "summary":          None,
            "simple_review":    None,
            "final_decision":   None,
            "errors":           [],
            "tokens_used":      {},
            "messages":         []
        }

        result_checkpoint = app_with_memory.invoke(checkpoint_state, config)

        print("✅ Run complete with checkpointing")
        decision_preview = (result_checkpoint.get('final_decision') or '')[:100]
        print(f"   Final decision: {decision_preview}...")
        print(f"\n💾 Checkpoints saved to: checkpoints.db")
        print("   Query with: sqlite3 checkpoints.db 'SELECT thread_id, checkpoint_id FROM checkpoints'")

        # Show the checkpoint history for this thread
        history = list(app_with_memory.get_state_history(config))
        print(f"\n📚 Checkpoint snapshots for thread 'pr-review-simple-001': {len(history)}")
        for snap in history[:5]:
            step   = snap.metadata.get('step', '?')
            source = snap.metadata.get('source', '?')
            keys   = [k for k, v in snap.values.items() if v]
            print(f"   Step {step:>2} ({source:>8}): populated fields = {keys}")

except ImportError as e:
    print(f"SqliteSaver not available: {e}")
    print("Install with: pip install langgraph[sqlite]")
except Exception as e:
    print(f"Checkpointer error: {e}")

## Step 21 — Try Your Own PR Diff (Interactive)

Edit `MY_DIFF` in the cell below and run it to review your own code change.

The graph will classify it, route it through the appropriate path, and return a structured verdict -- exactly as it would in a real CI/CD pipeline.

**Tips:**
- Paste a real git diff, or just a raw code snippet
- Include SQL queries, auth code, or secrets to trigger the full + BLOCKING path
- Use only a variable rename or import change to trigger the simple path

In [ ]:
import tempfile, os

# ✏️ EDIT THIS — paste your own PR diff or code snippet below
MY_DIFF = """\
diff --git a/app/api.py b/app/api.py
--- a/app/api.py
+++ b/app/api.py
@@ -10,6 +10,12 @@
 def get_user(user_id):
     return db.query(f"SELECT * FROM users WHERE id = {user_id}")

+def update_email(user_id, new_email):
+    db.execute(f"UPDATE users SET email = '{new_email}' WHERE id = {user_id}")
+    print(f"Updated email for user {user_id} to {new_email}")
+    return {"status": "ok"}
"""

# Write to a temp file and run through the graph
with tempfile.NamedTemporaryFile(mode='w', suffix='.diff', delete=False, dir='/tmp') as f:
    f.write(MY_DIFF)
    tmp_path = f.name

print("🚀 Running Code Review Tool on your custom diff")
print("=" * 60)

try:
    custom_state: PRReviewState = {
        "pr_file_path":     tmp_path,
        "pr_content":       "",
        "review_type":      "",
        "quality_findings": None,
        "security_findings": None,
        "summary":          None,
        "simple_review":    None,
        "final_decision":   None,
        "errors":           [],
        "tokens_used":      {},
        "messages":         []
    }
    result_custom = code_review_app.invoke(custom_state)
    print(f"\nRoute taken:  {result_custom.get('review_type', 'unknown')}")
    print(f"Total tokens: {result_custom.get('tokens_used', {}).get('total_tokens', 0):,}")
    print(f"\nMessage log:")
    for msg in result_custom['messages']:
        print(f"  {msg}")
finally:
    os.unlink(tmp_path)

## Next: Notebook 3 — Advanced Patterns

Open **`03_advanced_patterns.ipynb`** to explore:
- Unit testing nodes in isolation (no LLM calls needed)
- LLM-as-Judge pattern with `gpt-4o-mini` quality gating
- Retry loops, right-sizing models, and extensions